In [1]:
from langchain_huggingface import ChatHuggingFace, HuggingFacePipeline
from langgraph.graph import StateGraph, START, END
from typing import TypedDict

c:\lang\venv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1
c:\lang\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
llm = HuggingFacePipeline.from_model_id(model_id='Qwen/Qwen2.5-1.5B-Instruct', 
    task='text-generation',
    pipeline_kwargs=dict(
        temperature=0.7,
        max_new_tokens=250,
        max_length=None
    )
)
model = ChatHuggingFace(llm=llm)    


Loading weights: 100%|██████████| 338/338 [00:00<00:00, 1635.57it/s]
Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'max_length', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [3]:
class LLMState(TypedDict):
    question: str
    answer: str

In [4]:
def llm_qa(state: LLMState) -> LLMState:
    question = state['question']
    # Refined prompt for clarity and precision
    prompt = f"Provide a concise and clear answer to the following question: {question}"
    # Invoke the model with the refined prompt
    answer = model.invoke(prompt).content.strip()  # Strip unnecessary whitespace
    state['answer'] = answer
    return state

In [5]:
graph = StateGraph(LLMState)
graph.add_node('llm_qa',llm_qa)
graph.add_edge(START, 'llm_qa')
graph.add_edge('llm_qa', END)
workflow = graph.compile()

In [7]:
initial_state = {'question' : ' How far is sun from earth?'}
final_state =  workflow.invoke(initial_state)
print(final_state)
print(final_state['answer'])

{'question': ' How far is sun from earth?', 'answer': '<|im_start|>system\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\n<|im_start|>user\nProvide a concise and clear answer to the following question:  How far is sun from earth?<|im_end|>\n<|im_start|>assistant\nThe average distance between Earth and the Sun (also known as one Astronomical Unit or AU) is approximately 93 million miles or about 149.6 million kilometers. This distance remains relatively constant throughout the year due to their orbital paths around each other in nearly circular orbits.'}
<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
Provide a concise and clear answer to the following question:  How far is sun from earth?<|im_end|>
<|im_start|>assistant
The average distance between Earth and the Sun (also known as one Astronomical Unit or AU) is approximately 93 million miles or about 149.6 million kilometers. This dista